# K-Means — Program

This notebook contains the data workflow and runnable code. See [K-Means — Theory](03_k_means_theory.ipynb) for concepts.

**Task:** Clustering. **Data:** Generated two-dimensional blobs. Run top to bottom.

## 1. Load data

In [ ]:
from sklearn.datasets import make_blobs
import pandas as pd
X_array, _ = make_blobs(n_samples=400, centers=4, cluster_std=0.75, random_state=42)
X = pd.DataFrame(X_array, columns=["feature_1", "feature_2"])
y = None

## 2. Inspect raw data

In [ ]:
print("Feature shape:", X.shape)
print("Types:\n", X.dtypes)
print("Missing by column:\n", X.isna().sum())
print("Duplicate feature rows:", X.duplicated().sum())
print("Preview:\n", X.head())


## 3. Clean data
Drop duplicate observations and fill missing numeric values with each column median. The demo datasets are clean, but these checks and steps are useful for real data.

In [ ]:
import pandas as pd
if y is not None:
    combined = X.copy()
    combined["__target__"] = y
    combined = combined.drop_duplicates().reset_index(drop=True)
    y = combined.pop("__target__")
    X = combined
else:
    X = X.drop_duplicates().reset_index(drop=True)
X = X.apply(lambda col: col.fillna(col.median()) if col.isna().any() else col)
print("Clean feature shape:", X.shape)
print("Missing remaining:", int(X.isna().sum().sum()))
print("Duplicates remaining:", int(X.duplicated().sum()))

## 4. Analyze data
Review summary statistics and feature distributions. Classification also shows target balance.

In [ ]:
import matplotlib.pyplot as plt
print(X.describe().T[["mean", "std", "min", "max"]])
X.hist(figsize=(12, 8), bins=20)
plt.tight_layout(); plt.show()


## 5. Prepare training and test data
Use a fixed seed for repeatability. Class labels are stratified. Scaling is included in pipelines when needed.

In [ ]:
print("K-Means uses all unlabeled observations; there is no target or train/test split.")

## 6. Run the algorithm
Fit and evaluate the model. Metrics describe this example split; they are not guarantees of future performance.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
model = KMeans(n_clusters=4, n_init=10, random_state=42)
labels = model.fit_predict(X)
print(f"Inertia: {model.inertia_:.2f}")
print(f"Silhouette score: {silhouette_score(X, labels):.3f}")
plt.scatter(X.iloc[:, 0], X.iloc[:, 1], c=labels, cmap="viridis", s=25)
plt.scatter(model.cluster_centers_[:, 0], model.cluster_centers_[:, 1], c="red", marker="X", s=180, label="Centroids")
plt.title("K-Means clusters (k=4)"); plt.legend(); plt.show()

## 7. Inspect an example output

In [ ]:
print("First five cluster labels:", labels[:5])